# Spectroscopic reference set (part 2)

Real S-PLUS galaxies whose SFH labels come from **Bagpipes fits to their SDSS spectra under exactly the training prior** (`synthetic_model.py`), so they test the ML model on real data with labels it can match. Firefly (SDSS DR16) is an independent check of those fits, never a label.

**Inputs:** `dr3_corrected.csv`, `../Data/sdss_eboss_firefly-dr16.fits`, `../dustMaps/csfd/`, `training_set.parquet` (part 1), SDSS spectra (downloaded), S-PLUS Cloud aperture photometry (needs `SPLUS_USERNAME` / `SPLUS_PASSWORD`).
**Outputs:** `spectra/`, `spectral_fits/` (one `.h5` per fit), `spectral_fits/failures.csv`, `spectroscopic_reference.csv`, `firefly_comparison.png`.
**Run:** top to bottom with the *Inference4ETGs (MultiNest)* kernel, after `training_set.ipynb`. About 3 h; re-running skips finished fits, so raising `GALAXIES_PER_CELL` later only fits the new galaxies.

Spectra fit 3″ (SDSS) or 2″ (BOSS) fibres, so photometry is the 3″ S-PLUS aperture, and BOSS galaxies are reported separately. The **test** half of the split is used once, for the final ML evaluation; with ~30/30 galaxies this is a pilot-size set.

In [ ]:
import math
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.io import fits
from astroquery.sdss import SDSS
from dustmaps.csfd import CSFDQuery
from joblib import Parallel, delayed

import synthetic_model as sm

# Files and folders
SPLUS_FILE = "dr3_corrected.csv"
FIREFLY_FILE = "../Data/sdss_eboss_firefly-dr16.fits"
DUST_MAP_FILE = "../dustMaps/csfd/csfd_ebv.fits"
DUST_MASK_FILE = "../dustMaps/csfd/mask.fits"
TRAINING_FILE = "training_set.parquet"
SPECTRA_DIRECTORY = "spectra"
FITS_DIRECTORY = "spectral_fits"
PILOT_DIRECTORY = "spectral_fits/pilot"
FAILURES_FILE = "spectral_fits/failures.csv"
OUTPUT_FILE = "spectroscopic_reference.csv"
FIGURE_FILE = "firefly_comparison.png"
CREDENTIALS_FILE = "../.env"     # git-ignored; holds SPLUS_USERNAME and SPLUS_PASSWORD
APERTURE_CACHE_FILE = "splus_aperture_photometry.csv"   # raw 3" photometry of every S-PLUS x Firefly match

# Sample
MIN_PROB_GALAXY = 0.8
MATCH_RADIUS = 1.5               # [arcsec]
N_REDSHIFT_BINS = 4
N_AGE_BINS = 3
GALAXIES_PER_CELL = 5
N_REAL_GALAXIES = N_REDSHIFT_BINS * N_AGE_BINS * GALAXIES_PER_CELL   # 60
FIBER_DIAMETER = {"26": 3.0, "v5_13_0": 2.0}   # [arcsec] SDSS legacy and BOSS
SQL_CHUNK = 200                  # specobjids per SDSS SQL query

# Fits
N_WORKERS = 11
N_MOCKS = 6                      # 2 per SFH branch
MOCK_SFH_DRAWS = [0.25, 0.25, 0.75, 0.75, 0.99, 0.99]   # sfh_type draws: Dirichlet, skew-normal, + burst
MIN_MOCKS_RECOVERED = 5
MOCK_CHECK_LABELS = ["mw_age", "Z_fin", "f_old"]
FIRST_WAVE_REAL_GALAXIES = N_WORKERS - N_MOCKS            # 5 real galaxies share the first wave
BUDGET_HOURS = 3.0
TARGET_MINUTES_PER_FIT = 25.0
ITERATION_SAFETY = 0.8           # MultiNest slows down as it converges
PILOT_ITERATIONS = 1000
SEED = 0

# S-PLUS aperture photometry. DR4 keeps one table per band, each with its own RA/Dec columns
SPLUS_RA_COLUMN = "RA"
SPLUS_DEC_COLUMN = "DEC"
APERTURE_SOURCES = {}
for band in sm.BANDS:
    APERTURE_SOURCES[band] = (f"dr4_dual.dr4_dual_{band.lower()}", f"{band}_aper_3", f"e_{band}_aper_3")
SPLUS_QUERY_TIMEOUT = 600        # [s]
SPLUS_EXECUTION_MODE = "sync"    # small queries; async jobs are capped at 4 concurrent per account

SSFR_PASSIVE = -11.0             # [log yr^-1] star-forming above, passive below

os.makedirs(SPECTRA_DIRECTORY, exist_ok=True)
os.makedirs(PILOT_DIRECTORY, exist_ok=True)
failures = []

## 1. Sample
S-PLUS DR3 galaxies matched to Firefly DR16 spectra within 1.5″, with spectroscopic redshift inside the prior. One spectrum per S-PLUS galaxy (the closest match).

In [ ]:
# 1. S-PLUS galaxies
splus = pd.read_csv(SPLUS_FILE)
splus = splus[splus["PROB_GAL"] > MIN_PROB_GALAXY].reset_index(drop=True)

# 2. Firefly galaxies with a redshift inside the prior, in the S-PLUS declination range
FIREFLY_COLUMNS = ["SPECOBJID", "RUN2D", "PLATE", "MJD", "FIBERID", "RA", "DEC", "Z_NOQSO", "CLASS_NOQSO",
                   "Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"]
firefly_data = fits.getdata(FIREFLY_FILE, 1)
firefly = {}
for column in FIREFLY_COLUMNS:
    values = np.asarray(firefly_data[column])
    # FITS stores big-endian numbers; pandas needs the machine's native byte order
    firefly[column] = values.astype(values.dtype.newbyteorder("="))
firefly = pd.DataFrame(firefly)
firefly["CLASS_NOQSO"] = firefly["CLASS_NOQSO"].str.strip()
firefly["RUN2D"] = firefly["RUN2D"].str.strip()
is_galaxy = firefly["CLASS_NOQSO"] == "GALAXY"
in_prior = (firefly["Z_NOQSO"] > sm.Z_RANGE[0]) & (firefly["Z_NOQSO"] < sm.Z_RANGE[1])
in_footprint = (firefly["DEC"] > splus["DEC"].min() - 1) & (firefly["DEC"] < splus["DEC"].max() + 1)
firefly = firefly[is_galaxy & in_prior & in_footprint].reset_index(drop=True)
del firefly_data

# 3. Nearest S-PLUS galaxy of each spectrum
spectrum_coordinates = SkyCoord(firefly["RA"].to_numpy() * u.deg, firefly["DEC"].to_numpy() * u.deg)
splus_coordinates = SkyCoord(splus["RA"].to_numpy() * u.deg, splus["DEC"].to_numpy() * u.deg)
nearest_splus, separation, _ = match_coordinates_sky(spectrum_coordinates, splus_coordinates)
firefly["splus_index"] = nearest_splus
firefly["separation"] = separation.arcsec
matched = firefly[firefly["separation"] < MATCH_RADIUS]

# 4. One spectrum per S-PLUS galaxy: the closest
matched = matched.sort_values("separation").drop_duplicates("splus_index").reset_index(drop=True)
# Kept as text: eBOSS specobjids exceed the int64 range
matched["specobjid"] = matched["SPECOBJID"].str.strip()
matched["fiber_diameter"] = matched["RUN2D"].map(FIBER_DIAMETER)
for column in ("Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"):
    missing = matched[column] <= sm.FIREFLY_MISSING
    matched.loc[missing, column] = np.nan
print(f"{len(matched)} S-PLUS galaxies with an SDSS/BOSS spectrum; "
      f"{matched['fiber_diameter'].eq(3.0).sum()} SDSS, {matched['fiber_diameter'].eq(2.0).sum()} BOSS")

In [ ]:
# 1. Median S/N of each spectrum from the SDSS database
sn_by_id = {}
identifiers = matched["specobjid"].tolist()
for start in range(0, len(identifiers), SQL_CHUNK):
    chunk = identifiers[start:start + SQL_CHUNK]
    id_list = ", ".join(chunk)
    query = ("SELECT CAST(specObjID AS VARCHAR(20)) AS specobjid_text, snMedian "
             f"FROM SpecObjAll WHERE specObjID IN ({id_list})")
    result = SDSS.query_sql(query, data_release=16)
    for row in result:
        sn_by_id[str(row["specobjid_text"]).strip()] = float(row["snMedian"])
matched["sn_median"] = matched["specobjid"].map(sn_by_id)

# 2. Grid of redshift x Firefly age quantile bins; highest S/N first in each cell
candidates = matched.dropna(subset=["Chabrier_MILES_age_massW", "sn_median"]).copy()
candidates["redshift_bin"] = pd.qcut(candidates["Z_NOQSO"], N_REDSHIFT_BINS, labels=False)
candidates["age_bin"] = pd.qcut(candidates["Chabrier_MILES_age_massW"], N_AGE_BINS, labels=False)

chosen = []
for (redshift_bin, age_bin), cell in candidates.groupby(["redshift_bin", "age_bin"]):
    best = cell.sort_values("sn_median", ascending=False).head(GALAXIES_PER_CELL).copy()
    best["rank_in_cell"] = np.arange(len(best))
    chosen.append(best)
# Round-robin order (best of every cell first), so a budget cut never empties a whole cell
selected = pd.concat(chosen, ignore_index=True)
selected = selected.sort_values(["rank_in_cell", "redshift_bin", "age_bin"]).reset_index(drop=True)
selected = selected.rename(columns={"RA": "ra", "DEC": "dec", "Z_NOQSO": "z", "PLATE": "plate", "MJD": "mjd",
                                    "FIBERID": "fiberid", "Chabrier_MILES_age_massW": "firefly_age",
                                    "Chabrier_MILES_metallicity_massW": "firefly_metallicity",
                                    "Chabrier_MILES_total_mass": "firefly_mass"})
print(f"{len(selected)} galaxies selected (target {N_REAL_GALAXIES})")
display(pd.crosstab(selected["redshift_bin"], selected["age_bin"]))

## 2. Spectra
SDSS spectra are **not** corrected for Galactic extinction, so each is dereddened with the same map and law as the S-PLUS photometry (CSFD, CCM89, $R_V = 3.1$), converted to air wavelengths (the models' frame), masked (bad pixels, 5577/6300 Å sky lines) and rebinned to 5 Å.

In [ ]:
dust_map = CSFDQuery(map_fname=DUST_MAP_FILE, mask_fname=DUST_MASK_FILE)
galaxy_coordinates = SkyCoord(selected["ra"].to_numpy() * u.deg, selected["dec"].to_numpy() * u.deg)
# The map is a big-endian FITS image; float64 is native, which pandas needs
selected["ebv"] = dust_map(galaxy_coordinates).astype(float)

prepared_spectra = {}
for row in selected.itertuples():
    spectrum_file = os.path.join(SPECTRA_DIRECTORY, f"{row.specobjid}.fits")

    # 1. Download once
    if not os.path.exists(spectrum_file):
        try:
            downloaded = SDSS.get_spectra(plate=int(row.plate), mjd=int(row.mjd), fiberID=int(row.fiberid),
                                          data_release=16)
            downloaded[0].writeto(spectrum_file)
        except Exception as error:
            failures.append({"job_id": str(row.specobjid), "stage": "download", "error": repr(error)})
            continue

    # 2. Prepare for the fit
    with fits.open(spectrum_file) as spectrum_hdus:
        table = spectrum_hdus[1].data
        wavelength_vacuum = 10 ** table["loglam"].astype(float)
        flux = table["flux"].astype(float)
        ivar = table["ivar"].astype(float)
    try:
        prepared_spectra[row.specobjid] = sm.prepare_spectrum(wavelength_vacuum, flux, ivar, row.ebv)
    except ValueError as error:
        failures.append({"job_id": str(row.specobjid), "stage": "prepare", "error": repr(error)})

print(f"{len(prepared_spectra)} spectra ready; {len(failures)} failures so far")

## 3. Aperture photometry
3″-diameter S-PLUS magnitudes, matching the SDSS fibre, dereddened like `dr3_corrected.csv`. They set the mass normalisation of the labels and feed the coverage test; they never enter the spectral fit.

The raw (not dereddened) magnitudes of every S-PLUS x Firefly match are cached in `splus_aperture_photometry.csv`; the S-PLUS Cloud is queried only for selected galaxies missing from that cache.

In [ ]:
# Output column names, built once
aperture_columns = []
aperture_error_columns = []
for band in sm.BANDS:
    aperture_columns.append(f"{band}_aper")
    aperture_error_columns.append(f"e_{band}_aper")


def query_to_dataframe(result):
    """S-PLUS Cloud query result as a DataFrame."""
    if hasattr(result, "to_pandas"):
        return result.to_pandas()
    return pd.DataFrame(result)


def cone_query(table, columns, ra, dec, radius_arcsec):
    """ADQL selecting columns of every object within radius of one position.

    One position per query: the S-PLUS Cloud rejects several OR-ed cones with "Column 'ra' does not exist".
    """
    radius_degrees = radius_arcsec / 3600
    column_list = ", ".join([SPLUS_RA_COLUMN, SPLUS_DEC_COLUMN, *columns])
    return (f"SELECT {column_list} FROM {table} WHERE 1 = CONTAINS(POINT('ICRS', {SPLUS_RA_COLUMN}, "
            f"{SPLUS_DEC_COLUMN}), CIRCLE('ICRS', {ra}, {dec}, {radius_degrees}))")


def query_aperture_photometry(galaxies):
    """Raw 3" aperture magnitudes and errors of galaxies, from the S-PLUS Cloud.

    Inputs: galaxies: DataFrame with columns ra and dec [deg].
    Returns: DataFrame indexed like galaxies, with {band}_aper and e_{band}_aper columns (NaN if not found).
    """
    import splusdata
    from dotenv import load_dotenv

    load_dotenv(CREDENTIALS_FILE)
    connection = splusdata.Core(os.environ["SPLUS_USERNAME"], os.environ["SPLUS_PASSWORD"])

    photometry = pd.DataFrame(np.nan, index=galaxies.index, columns=aperture_columns + aperture_error_columns)
    for galaxy in galaxies.itertuples():
        galaxy_position = SkyCoord(galaxy.ra * u.deg, galaxy.dec * u.deg)
        # DR4 keeps one table per band, so one query per band
        for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
            query = cone_query(table, [magnitude_column, error_column], galaxy.ra, galaxy.dec, MATCH_RADIUS)
            result = query_to_dataframe(connection.query(query, execution_mode=SPLUS_EXECUTION_MODE,
                                                         timeout=SPLUS_QUERY_TIMEOUT))
            if len(result) == 0:
                continue
            result_positions = SkyCoord(result[SPLUS_RA_COLUMN].to_numpy() * u.deg,
                                        result[SPLUS_DEC_COLUMN].to_numpy() * u.deg)
            closest = np.argmin(result_positions.separation(galaxy_position).arcsec)
            photometry.loc[galaxy.Index, f"{band}_aper"] = result[magnitude_column].iloc[closest]
            photometry.loc[galaxy.Index, f"e_{band}_aper"] = result[error_column].iloc[closest]
    return photometry


# 1. Raw magnitudes from the cache, by specobjid
for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
    selected[f"{band}_aper"] = np.nan
    selected[f"e_{band}_aper"] = np.nan
in_cache = pd.Series(False, index=selected.index)
if os.path.exists(APERTURE_CACHE_FILE):
    cache = pd.read_csv(APERTURE_CACHE_FILE, dtype={"specobjid": str})
    cache = cache.drop_duplicates("specobjid").set_index("specobjid")
    in_cache = selected["specobjid"].isin(cache.index)
    cached = cache.loc[selected.loc[in_cache, "specobjid"]]
    for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
        selected.loc[in_cache, f"{band}_aper"] = cached[magnitude_column].to_numpy()
        selected.loc[in_cache, f"e_{band}_aper"] = cached[error_column].to_numpy()
print(f"{in_cache.sum()} of {len(selected)} galaxies found in {APERTURE_CACHE_FILE}")

# 2. The rest from the S-PLUS Cloud
if not in_cache.all():
    queried = query_aperture_photometry(selected[~in_cache])
    selected.loc[queried.index, queried.columns] = queried

# 3. Galactic extinction, as for dr3_corrected.csv (non-detections keep their 99)
raw_magnitudes = selected[aperture_columns].to_numpy()
dereddened = sm.deredden_magnitudes(raw_magnitudes, selected["ebv"].to_numpy())
measured = raw_magnitudes < sm.UNDETECTED_MAGNITUDE
selected[aperture_columns] = np.where(measured, dereddened, raw_magnitudes)

missing_photometry = selected["r_aper"].isna()
print(f"Aperture photometry found for {(~missing_photometry).sum()} of {len(selected)} galaxies")
assert (~missing_photometry).sum() >= 0.9 * len(selected), "Aperture photometry missing for >10% of galaxies"

## 4. Fits
**Pilot:** one short fit measures the time per MultiNest iteration, which sets the iteration cap for ~25 min fits. **Inject-and-recover:** 6 mocks (2 per SFH branch) with real wavelengths, masks and errors run in the first wave with 5 real galaxies; at least 5 must recover the true `mw_age`, `Z_fin` and `f_old` inside their 95% intervals before the remaining fits start. Fits that hit the cap are flagged `converged = False`.

In [ ]:
rng = np.random.default_rng(SEED)
pilot_galaxy = next(iter(prepared_spectra))
pilot_job = {"job_id": f"pilot_{pilot_galaxy}", "spectrum": prepared_spectra[pilot_galaxy],
             "redshift": float(selected.loc[selected["specobjid"] == pilot_galaxy, "z"].iloc[0])}
pilot = sm.fit_and_save(pilot_job, PILOT_DIRECTORY, PILOT_ITERATIONS)
assert pilot["status"] in ("fitted", "skipped"), pilot["error"]

seconds_per_iteration = pilot["run_time"] / pilot["n_iterations"]
max_iterations = int(ITERATION_SAFETY * TARGET_MINUTES_PER_FIT * 60 / seconds_per_iteration)
n_fits = N_MOCKS + len(prepared_spectra)
n_waves = math.ceil(n_fits / N_WORKERS)
print(f"Pilot: {seconds_per_iteration:.2f} s per iteration -> cap of {max_iterations} iterations per fit")
print(f"Projected: {n_fits} fits in {n_waves} waves of ~{TARGET_MINUTES_PER_FIT:.0f} min "
      f"= {n_waves * TARGET_MINUTES_PER_FIT / 60:.1f} h (budget {BUDGET_HOURS} h)")

In [ ]:
def run_wave(jobs):
    """Fit a list of jobs on N_WORKERS processes; returns their summaries."""
    return Parallel(n_jobs=N_WORKERS, verbose=5)(
        delayed(sm.fit_and_save)(job, FITS_DIRECTORY, max_iterations) for job in jobs
    )


def real_galaxy_job(specobjid):
    """Fit job of one selected galaxy."""
    redshift = float(selected.loc[selected["specobjid"] == specobjid, "z"].iloc[0])
    return {"job_id": str(specobjid), "spectrum": prepared_spectra[specobjid], "redshift": redshift}


# 1. Mock jobs: real wavelengths, masks and errors; known unit-cube points
real_identifiers = list(prepared_spectra)
mock_jobs = []
mock_truths = {}
sfh_type_index = sm.SPECTRAL_DIMS.index("sfh_type")
for mock_index, sfh_type_draw in enumerate(MOCK_SFH_DRAWS):
    template_identifier = real_identifiers[rng.integers(len(real_identifiers))]
    template_job = real_galaxy_job(template_identifier)
    cube = rng.random(len(sm.SPECTRAL_DIMS))
    cube[sfh_type_index] = sfh_type_draw
    mock_spectrum, truth = sm.make_mock_spectrum(cube, template_job["redshift"], template_job["spectrum"], rng)
    job_id = f"mock_{mock_index}"
    mock_jobs.append({"job_id": job_id, "spectrum": mock_spectrum, "redshift": template_job["redshift"]})
    mock_truths[job_id] = truth

# 2. First wave
fitting_start = time.time()
first_wave_jobs = list(mock_jobs)
for specobjid in real_identifiers[:FIRST_WAVE_REAL_GALAXIES]:
    first_wave_jobs.append(real_galaxy_job(specobjid))
fit_summaries = run_wave(first_wave_jobs)
first_wave_hours = (time.time() - fitting_start) / 3600

# 3. Inject-and-recover: truth inside the 95% interval for every checked label
recovered = 0
for job in mock_jobs:
    posterior, _ = sm.load_fit(os.path.join(FITS_DIRECTORY, f"{job['job_id']}.h5"))
    inside_all = True
    for label in MOCK_CHECK_LABELS:
        low, high = np.nanpercentile(posterior[label], [2.5, 97.5])
        truth_value = mock_truths[job["job_id"]][label]
        inside = low <= truth_value <= high
        inside_all = inside_all and inside
        print(f"{job['job_id']} {label}: truth {truth_value:.3f}, 95% [{low:.3f}, {high:.3f}] {'ok' if inside else 'MISS'}")
    recovered += int(inside_all)
print(f"{recovered} of {len(mock_jobs)} mocks recovered; first wave took {first_wave_hours:.2f} h")
assert recovered >= MIN_MOCKS_RECOVERED, "Inject-and-recover failed: do not trust the real fits; investigate first"

In [ ]:
# Fit as many of the remaining galaxies as the budget allows, at the measured wave time
remaining = real_identifiers[FIRST_WAVE_REAL_GALAXIES:]
hours_left = BUDGET_HOURS - (time.time() - fitting_start) / 3600
# On a re-run the first wave is skipped and takes no time; fall back to the target fit time
if first_wave_hours < 0.05:
    wave_hours = TARGET_MINUTES_PER_FIT / 60
else:
    wave_hours = first_wave_hours
waves_left = int(hours_left // wave_hours)
n_allowed = waves_left * N_WORKERS
if n_allowed < len(remaining):
    print(f"Budget allows {n_allowed} of the {len(remaining)} remaining galaxies; "
          f"fitting the first {n_allowed}. Raise BUDGET_HOURS and re-run to fit the rest.")
    remaining = remaining[:n_allowed]

remaining_jobs = []
for specobjid in remaining:
    remaining_jobs.append(real_galaxy_job(specobjid))
fit_summaries = fit_summaries + run_wave(remaining_jobs)

summary_table = pd.DataFrame(fit_summaries)
for record in fit_summaries:
    if record["status"] == "failed":
        failures.append({"job_id": record["job_id"], "stage": "fit", "error": record["error"]})
pd.DataFrame(failures).to_csv(FAILURES_FILE, index=False)
print(f"Fitting took {(time.time() - fitting_start) / 3600:.2f} h")
display(summary_table["status"].value_counts())
display(summary_table.groupby("status")[["run_time", "n_iterations"]].median())

## 5. Labels, Firefly check, coverage and split
Each posterior sample's mass-like labels are rescaled by its aperture $r$ offset ($-0.4\,\Delta m$ dex), then summarised as 16/50/84th percentiles. A fit is **Firefly-consistent** if mass-weighted age and metallicity agree within 0.3 dex; failures are flagged, not deleted. Coverage uses the aperture photometry at the spectroscopic redshift. Galaxies that converged, are covered and are Firefly-consistent go to `validation` or `test` by a hash of `specobjid`. Read the output with `dtype={"specobjid": str}` (eBOSS ids exceed int64).

In [ ]:
LABEL_COLUMNS = ["log_ssfr", "mw_metallicity", "veldisp", "noise_scale", "Z_fin", "Av_ism",
                 *sm.SUMMARY_LABEL_COLUMNS, *sm.BIN_LABEL_COLUMNS]

# 1. Posterior summaries per galaxy
rows = []
for galaxy in selected.itertuples():
    fit_file = os.path.join(FITS_DIRECTORY, f"{galaxy.specobjid}.h5")
    row = {"specobjid": galaxy.specobjid}
    if not os.path.exists(fit_file):
        row["fit_status"] = "not_fitted"
        rows.append(row)
        continue
    posterior, metadata = sm.load_fit(fit_file)

    # Mass-like labels follow the aperture r magnitude: log X = log X_model - 0.4 (r_obs - r_model)
    offsets = galaxy.r_aper - posterior["model_r"]
    for label in sm.MASS_DEPENDENT_LABELS:
        posterior[label] = posterior[label] - 0.4 * offsets
    posterior["log_ssfr"] = posterior["log_sfr"] - posterior["log_mstar"]   # normalisation-free

    row["fit_status"] = "fitted"
    row["converged"] = bool(metadata["converged"])
    row["fit_hours"] = metadata["run_time"] / 3600
    row.update(sm.summarise_posterior(posterior, LABEL_COLUMNS))
    rows.append(row)
reference = selected.merge(pd.DataFrame(rows), on="specobjid", how="left")

# 2. Firefly check
firefly_check = sm.compare_with_firefly(
    reference["mw_age_median"].to_numpy(), reference["mw_metallicity_median"].to_numpy(),
    reference["firefly_age"].to_numpy(), reference["firefly_metallicity"].to_numpy(),
)
for column in firefly_check.columns:
    reference[column] = firefly_check[column].to_numpy()
reference["delta_log_mass"] = reference["log_mstar_median"] - np.log10(reference["firefly_mass"])

# 3. Coverage of the aperture photometry at the spectroscopic redshift
training = pd.read_parquet(TRAINING_FILE, columns=["redshift", *sm.BANDS])
aperture_magnitudes = reference[aperture_columns].to_numpy()
aperture_errors = reference[aperture_error_columns].to_numpy()
aperture_detected = sm.detected_bands(aperture_magnitudes, aperture_errors)
coverage = sm.coverage_test(aperture_magnitudes, aperture_errors, aperture_detected,
                            reference["z"].to_numpy(), np.full(len(reference), sm.DZ_MIN),
                            training[sm.BANDS].to_numpy(), training["redshift"].to_numpy())
for column in coverage.columns:
    reference[column] = coverage[column].to_numpy()

# 4. Locked split
usable = (reference["converged"].fillna(False).astype(bool)
          & reference["covered"].astype(bool)
          & reference["firefly_consistent"].fillna(False).astype(bool))
reference["split"] = "excluded"
for row_index in np.flatnonzero(usable.to_numpy()):
    reference.loc[row_index, "split"] = sm.split_by_hash(reference.loc[row_index, "specobjid"])

reference.to_csv(OUTPUT_FILE, index=False)
fitted = reference["fit_status"] == "fitted"
assert reference.loc[fitted, "mw_age_median"].notna().all(), "A fitted galaxy has no labels in the output"
print(f"{OUTPUT_FILE}: {len(reference)} galaxies, {fitted.sum()} fitted, "
      f"{reference['converged'].fillna(False).sum()} converged, {reference['covered'].sum()} covered, "
      f"{reference['firefly_consistent'].fillna(False).sum()} Firefly-consistent")
display(reference["split"].value_counts())

In [ ]:
reference["fibre"] = reference["fiber_diameter"].map({3.0: "SDSS 3\"", 2.0: "BOSS 2\""})
reference["galaxy_type"] = np.where(reference["log_ssfr_median"] > SSFR_PASSIVE, "star-forming", "passive")

for grouping in ("fibre", "galaxy_type"):
    grouped = reference[fitted].groupby(grouping)
    table = pd.DataFrame({
        "median Δlog age": grouped["delta_log_age"].median(),
        "scatter Δlog age": grouped["delta_log_age"].std(),
        "median Δlog Z": grouped["delta_log_metallicity"].median(),
        "scatter Δlog Z": grouped["delta_log_metallicity"].std(),
        "median Δlog M": grouped["delta_log_mass"].median(),
        "consistent": grouped["firefly_consistent"].mean(),
        "galaxies": grouped.size(),
    })
    display(table.round(3))

figure, axes = plt.subplots(1, 2, figsize=(12, 5.5))
comparisons = [
    (np.log10(reference["firefly_age"]), np.log10(reference["mw_age_median"] * sm.GYR_TO_YEARS), "log age [yr]"),
    (np.log10(reference["firefly_metallicity"]), np.log10(reference["mw_metallicity_median"]), r"log $Z/Z_\odot$"),
]
for axis, (firefly_values, bagpipes_values, label) in zip(axes, comparisons):
    for galaxy_type, marker in (("star-forming", "o"), ("passive", "s")):
        chosen = fitted & (reference["galaxy_type"] == galaxy_type)
        axis.scatter(firefly_values[chosen], bagpipes_values[chosen], marker=marker, label=galaxy_type)
    limits = [np.nanmin([firefly_values, bagpipes_values]), np.nanmax([firefly_values, bagpipes_values])]
    axis.plot(limits, limits, color="black", lw=1)
    axis.set(xlabel=f"Firefly {label}", ylabel=f"Bagpipes {label}")
    axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_FILE, dpi=200)
plt.show()

## Next pass
Raise `GALAXIES_PER_CELL` (and `BUDGET_HOURS`) and re-run: finished fits are skipped and the hash split never moves an existing galaxy. Add DR4 Stripe 82 galaxies by pointing `SPLUS_FILE` at a DR4 catalogue with the same columns.